[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/experimental-psychology/blob/main/notebooks/stats_refresher.ipynb)

# Stats Refresher: A Worked Example

### PSYC 11: Laboratory in Psychological Science

This notebook walks through the core statistical tests you'll encounter in this course, all applied to a single (simulated) dataset. The goal is to build intuitions about **when** and **why** to use each test — not to memorize formulas.

**Scenario:** We ran a study on **sleep and academic performance** in 120 college students. Each student reported:
- Hours of sleep the night before the exam
- Daily caffeine intake (mg)
- Hours spent studying
- Study method used (active recall vs. passive review)
- Whether they pulled an all-nighter (yes/no)
- Their exam score (0–100)
- Their class year (first-year, sophomore, junior, senior)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_theme(style='whitegrid', font_scale=1.1)

## 1. Generate the dataset

We'll simulate data with realistic correlations built in. Sleep and caffeine are inversely related (people who sleep less drink more coffee), and exam scores depend on sleep, study hours, and study method.

In [ ]:
n = 120

# Class year (unequal sizes — more first-years)
class_year = np.random.choice(
    ['First-year', 'Sophomore', 'Junior', 'Senior'],
    size=n,
    p=[0.35, 0.30, 0.20, 0.15]
)

# Study method (roughly balanced)
study_method = np.random.choice(['Active recall', 'Passive review'], size=n)

# Sleep hours: normally distributed, mean=6.5, sd=1.5, clipped to [2, 10]
sleep_hours = np.clip(np.random.normal(6.5, 1.5, n), 2, 10)

# Caffeine: inversely related to sleep + noise
caffeine_mg = np.clip(400 - 30 * sleep_hours + np.random.normal(0, 40, n), 0, 600).astype(int)

# Study hours: 1–12, right-skewed
study_hours = np.clip(np.random.exponential(4, n) + 1, 1, 12).round(1)

# All-nighter: more likely if sleep < 5
all_nighter_prob = np.where(sleep_hours < 5, 0.65, 0.18)
all_nighter = np.random.binomial(1, all_nighter_prob).astype(bool)

# Exam score: depends on sleep, study hours, method, and noise
method_bonus = np.where(study_method == 'Active recall', 5, 0)
exam_score = np.clip(
    40 + 3 * sleep_hours + 2.5 * study_hours + method_bonus
    - 8 * all_nighter + np.random.normal(0, 8, n),
    0, 100
).round(1)

df = pd.DataFrame({
    'class_year': class_year,
    'study_method': study_method,
    'sleep_hours': sleep_hours.round(1),
    'caffeine_mg': caffeine_mg,
    'study_hours': study_hours,
    'all_nighter': all_nighter,
    'exam_score': exam_score
})

print(f'Dataset: {len(df)} students')
df.head(10)

In [ ]:
df.describe()

---

## 2. Hypothesis Testing & Null Distributions

Before diving into specific tests, let's visualize **what a null distribution looks like** and what a p-value actually measures.

**Key idea:** The null distribution is the distribution of a test statistic you'd expect to see *if there were no real effect*. The p-value is the probability of getting a result at least as extreme as yours under that null.

In [ ]:
# Permutation-based null distribution for the difference in exam scores
# between active recall and passive review groups
active = df[df['study_method'] == 'Active recall']['exam_score']
passive = df[df['study_method'] == 'Passive review']['exam_score']
observed_diff = active.mean() - passive.mean()

# Generate null distribution by shuffling labels 10,000 times
n_perms = 10_000
null_diffs = np.zeros(n_perms)
scores = df['exam_score'].values
n_active = len(active)

for i in range(n_perms):
    shuffled = np.random.permutation(scores)
    null_diffs[i] = shuffled[:n_active].mean() - shuffled[n_active:].mean()

perm_p = np.mean(np.abs(null_diffs) >= np.abs(observed_diff))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(null_diffs, bins=50, color='steelblue', alpha=0.7, edgecolor='white',
        label='Null distribution')
ax.axvline(observed_diff, color='red', linewidth=2, linestyle='--',
           label=f'Observed diff = {observed_diff:.2f}')
ax.axvline(-observed_diff, color='red', linewidth=2, linestyle='--', alpha=0.5)
ax.set_xlabel('Difference in means (Active − Passive)')
ax.set_ylabel('Count')
ax.set_title(f'Null Distribution (permutation test)\np = {perm_p:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Observed difference: {observed_diff:.2f} points')
print(f'Permutation p-value (two-tailed): {perm_p:.4f}')

---

## 3. T-Test

**When to use:** Comparing the means of **two groups** on a continuous outcome.

**Question:** Do students who use active recall score differently on the exam than those who use passive review?

In [ ]:
# Independent samples t-test
t_stat, t_p = stats.ttest_ind(active, passive)

# Effect size: Cohen's d
pooled_std = np.sqrt(((len(active) - 1) * active.std()**2 +
                       (len(passive) - 1) * passive.std()**2) /
                      (len(active) + len(passive) - 2))
cohens_d = (active.mean() - passive.mean()) / pooled_std

print(f'Active recall:  M = {active.mean():.1f}, SD = {active.std():.1f}, n = {len(active)}')
print(f'Passive review: M = {passive.mean():.1f}, SD = {passive.std():.1f}, n = {len(passive)}')
print(f'\nt({len(active) + len(passive) - 2}) = {t_stat:.3f}, p = {t_p:.4f}')
print(f"Cohen's d = {cohens_d:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='study_method', y='exam_score', hue='study_method',
            ax=ax, palette='Set2', legend=False)
sns.stripplot(data=df, x='study_method', y='exam_score', ax=ax,
              color='black', alpha=0.3, size=4, legend=False)
ax.set_xlabel('Study Method')
ax.set_ylabel('Exam Score')
ax.set_title(f't-test: Active Recall vs. Passive Review\nt = {t_stat:.2f}, p = {t_p:.4f}, d = {cohens_d:.2f}')
plt.tight_layout()
plt.show()

---

## 4. ANOVA (Analysis of Variance)

**When to use:** Comparing the means of **three or more groups** on a continuous outcome.

**Question:** Do exam scores differ across class years (first-year, sophomore, junior, senior)?

ANOVA is essentially an extension of the t-test. Instead of asking "are these two means different?" it asks "are *any* of these group means different from the others?"

In [ ]:
# One-way ANOVA
groups = [group['exam_score'].values for _, group in df.groupby('class_year')]
f_stat, anova_p = stats.f_oneway(*groups)

# Effect size: eta-squared
grand_mean = df['exam_score'].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
ss_total = sum((df['exam_score'] - grand_mean)**2)
eta_sq = ss_between / ss_total

print('Group means:')
for year in ['First-year', 'Sophomore', 'Junior', 'Senior']:
    subset = df[df['class_year'] == year]['exam_score']
    print(f'  {year}: M = {subset.mean():.1f}, SD = {subset.std():.1f}, n = {len(subset)}')

k = len(groups)
print(f'\nF({k - 1}, {n - k}) = {f_stat:.3f}, p = {anova_p:.4f}')
print(f'η² = {eta_sq:.4f}')

fig, ax = plt.subplots(figsize=(9, 5))
order = ['First-year', 'Sophomore', 'Junior', 'Senior']
sns.boxplot(data=df, x='class_year', y='exam_score', hue='class_year',
            order=order, ax=ax, palette='Set3', legend=False)
sns.stripplot(data=df, x='class_year', y='exam_score', order=order,
              ax=ax, color='black', alpha=0.3, size=4, legend=False)
ax.set_xlabel('Class Year')
ax.set_ylabel('Exam Score')
ax.set_title(f'One-Way ANOVA: Exam Score by Class Year\nF = {f_stat:.2f}, p = {anova_p:.4f}, η² = {eta_sq:.3f}')
plt.tight_layout()
plt.show()

---

## 5. Correlation

**When to use:** Measuring the **linear relationship** between two continuous variables.

**Question:** Is there a relationship between sleep hours and exam scores?

In [ ]:
# Pearson correlation
r, corr_p = stats.pearsonr(df['sleep_hours'], df['exam_score'])

print(f'Pearson r = {r:.3f}, p = {corr_p:.6f}')
print(f'r² = {r**2:.3f} (proportion of variance explained)')

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df['sleep_hours'], df['exam_score'], alpha=0.5, color='steelblue', s=40)

# Add regression line
z = np.polyfit(df['sleep_hours'], df['exam_score'], 1)
x_line = np.linspace(df['sleep_hours'].min(), df['sleep_hours'].max(), 100)
ax.plot(x_line, np.polyval(z, x_line), 'r-', linewidth=2)

ax.set_xlabel('Sleep Hours')
ax.set_ylabel('Exam Score')
ax.set_title(f'Sleep vs. Exam Score\nr = {r:.3f}, p = {corr_p:.4f}')
plt.tight_layout()
plt.show()

---

## 6. Regression

**When to use:** Predicting a continuous outcome from one or more predictors. Regression extends correlation by letting you:
1. Make **predictions** (what score would you expect for someone who slept 8 hours and studied 6?)
2. Control for **multiple variables** simultaneously

**Question:** How well can we predict exam scores from sleep hours, study hours, and caffeine intake together?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Multiple regression
predictors = ['sleep_hours', 'study_hours', 'caffeine_mg']
X = df[predictors].values
y = df['exam_score'].values

model = LinearRegression().fit(X, y)
y_pred = model.predict(X)

print('Multiple regression: exam_score ~ sleep_hours + study_hours + caffeine_mg')
print(f'\nIntercept: {model.intercept_:.2f}')
for name, coef in zip(predictors, model.coef_):
    print(f'  {name}: β = {coef:.3f}')
print(f'\nR² = {r2_score(y, y_pred):.3f}')

# Visualize predicted vs. actual
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y, y_pred, alpha=0.5, color='steelblue', s=40)
lims = [min(y.min(), y_pred.min()) - 2, max(y.max(), y_pred.max()) + 2]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Exam Score')
ax.set_ylabel('Predicted Exam Score')
ax.set_title(f'Multiple Regression: Predicted vs. Actual\nR² = {r2_score(y, y_pred):.3f}')
ax.legend()
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

**Think about it:** Notice that the caffeine coefficient is small — even though caffeine and exam scores *are* correlated. Why? Because caffeine is also correlated with sleep, and regression controls for that overlap. This is one of the most important differences between correlation and regression.

---

## 7. Chi-Squared Test

**When to use:** Testing whether two **categorical variables** are associated (i.e., whether they are independent or not).

**Question:** Is there an association between study method (active recall vs. passive review) and whether students pulled an all-nighter?

In [ ]:
# Create contingency table
ct = pd.crosstab(df['study_method'], df['all_nighter'],
                 margins=True, margins_name='Total')
ct.columns = ['No All-Nighter', 'All-Nighter', 'Total']
print('Contingency Table:')
print(ct)
print()

# Chi-squared test (without margins)
ct_raw = pd.crosstab(df['study_method'], df['all_nighter'])
chi2, chi_p, dof, expected = stats.chi2_contingency(ct_raw)

# Effect size: Cramér's V
cramers_v = np.sqrt(chi2 / (n * (min(ct_raw.shape) - 1)))

print(f'χ²({dof}) = {chi2:.3f}, p = {chi_p:.4f}')
print(f"Cramér's V = {cramers_v:.3f}")
print(f'\nExpected frequencies (if independent):')
print(pd.DataFrame(expected, index=ct_raw.index, columns=['No All-Nighter', 'All-Nighter']).round(1))

---

## 8. Binomial Test

**When to use:** Testing whether a **proportion** differs from a hypothesized value.

**Question:** Previous research suggests that about 20% of college students pull all-nighters before exams. Is the rate in our sample different from 20%?

In [ ]:
# Binomial test
n_allnighters = df['all_nighter'].sum()
n_total = len(df)
observed_prop = n_allnighters / n_total
hypothesized_prop = 0.20

binom_result = stats.binomtest(n_allnighters, n_total, hypothesized_prop)

print(f'Observed: {n_allnighters}/{n_total} = {observed_prop:.1%} pulled an all-nighter')
print(f'Hypothesized proportion: {hypothesized_prop:.0%}')
print(f'\nBinomial test p-value: {binom_result.pvalue:.4f}')
print(f'95% CI for proportion: [{binom_result.proportion_ci().low:.3f}, {binom_result.proportion_ci().high:.3f}]')

---

## 9. Confidence Intervals

**What they tell you:** A 95% CI gives a range of plausible values for the true population parameter. If you repeated the study many times, about 95% of the CIs would contain the true value.

**Common misconception:** The CI does *not* mean there's a 95% probability the true value is inside it. The true value is fixed — the interval is the random part.

Let's compute CIs for the mean exam score in each study method group.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

methods = ['Active recall', 'Passive review']
means = []
cis = []

for method in methods:
    scores = df[df['study_method'] == method]['exam_score']
    m = scores.mean()
    se = stats.sem(scores)
    ci = stats.t.interval(0.95, df=len(scores) - 1, loc=m, scale=se)
    means.append(m)
    cis.append(ci)
    print(f'{method}: M = {m:.1f}, 95% CI = [{ci[0]:.1f}, {ci[1]:.1f}]')

# Plot
colors = ['#4CAF50', '#FF9800']
for i, method in enumerate(methods):
    ax.barh(i, means[i], color=colors[i], alpha=0.7, height=0.5)
    ax.errorbar(means[i], i, xerr=[[means[i] - cis[i][0]], [cis[i][1] - means[i]]],
                fmt='o', color='black', capsize=8, capthick=2)

ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods)
ax.set_xlabel('Exam Score')
ax.set_title('Mean Exam Score by Study Method (with 95% CIs)')
plt.tight_layout()
plt.show()

---

## 10. P-Values & Effect Sizes: The Full Picture

Let's compare all of our results side by side to see the interplay between **statistical significance** (p-value) and **practical significance** (effect size).

In [ ]:
summary = pd.DataFrame({
    'Test': ['t-test (study method)', 'ANOVA (class year)', 'Correlation (sleep × score)',
             'Regression (multi)', 'Chi-squared (method × all-nighter)', 'Binomial (all-nighter rate)'],
    'Statistic': [f't = {t_stat:.3f}', f'F = {f_stat:.3f}', f'r = {r:.3f}',
                  f'R² = {r2_score(y, y_pred):.3f}', f'χ² = {chi2:.3f}',
                  f'prop = {observed_prop:.3f}'],
    'p-value': [f'{t_p:.4f}', f'{anova_p:.4f}', f'{corr_p:.6f}',
                '—', f'{chi_p:.4f}', f'{binom_result.pvalue:.4f}'],
    'Effect Size': [f"d = {cohens_d:.3f}", f'η² = {eta_sq:.4f}', f'r² = {r**2:.3f}',
                    f'R² = {r2_score(y, y_pred):.3f}', f"V = {cramers_v:.3f}",
                    f'diff = {observed_prop - hypothesized_prop:+.1%}']
})

print(summary.to_string(index=False))

---

## Summary: Choosing the Right Test

| Situation | Variables | Test | Effect Size |
|-|-|-|-|
| Compare 2 group means | 1 categorical (2 levels), 1 continuous | **t-test** | Cohen's d |
| Compare 3+ group means | 1 categorical (3+ levels), 1 continuous | **ANOVA** | η² |
| Linear association | 2 continuous | **Correlation** | r, r² |
| Predict from multiple vars | Multiple predictors, 1 continuous outcome | **Regression** | R² |
| Association of categories | 2 categorical | **Chi-squared** | Cramér's V |
| Test a proportion | 1 binary outcome | **Binomial test** | Observed − expected |
| Range of plausible values | Any parameter estimate | **Confidence interval** | (width of CI) |